In [1]:
import py7zr
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
ZIP_DIR = Path('zips')
RAW_DIR = Path('data/raw')

def unzip_all(zip_dir, raw_dir):
    for zip_path in sorted(zip_dir.glob('*.7z')):
        year = re.search(r'20\d{2}', zip_path.name).group()
        out_dir = raw_dir / year
        out_dir.mkdir(parents=True, exist_ok=True)
        with py7zr.SevenZipFile(zip_path, 'r') as z:
            z.extractall(out_dir)
        print(f'{zip_path.name}  ->  {out_dir}')

unzip_all(ZIP_DIR, RAW_DIR)

In [6]:
txt_files = sorted(RAW_DIR.rglob('*.txt'))
print('Number of .txt files:', len(txt_files))
for f in txt_files:
    print(f, '|', round(f.stat().st_size / 1e6, 1), 'MB')

Number of .txt files: 0


In [7]:
with open(txt_files[0]) as f:
    print(f.readline())
    print(f.readline())

IndexError: list index out of range

In [8]:
def read_header(path):
    with open(path) as f:
        return [c.strip().strip('[]').lower() for c in f.readline().strip().split(',')]

ref = read_header(txt_files[0])
print(len(ref), 'columns:', ref)

for f in txt_files:
    if read_header(f) != ref:
        print('DIFFERENT:', f.name)
print('Check done')

IndexError: list index out of range

In [9]:
KEEP = ['quote_date', 'underlying_last', 'expire_date', 'dte', 'strike',
        'c_bid', 'c_ask', 'c_last', 'c_volume', 'c_size',
        'c_iv', 'c_delta', 'c_gamma', 'c_vega', 'c_theta', 'c_rho']

def load_calls(path):
    return pd.read_csv(path,
                       names=ref, header=0, 
                       usecols=KEEP,
                       skipinitialspace=True,     
                       low_memory=False)

calls = pd.concat([load_calls(f) for f in txt_files], ignore_index=True)
calls['quote_date'] = pd.to_datetime(calls['quote_date'])
calls['expire_date'] = pd.to_datetime(calls['expire_date'])

print(calls.shape)
print(calls.dtypes)
calls.head()

ValueError: No objects to concatenate

In [10]:

calls['mid'] = (calls['c_bid'] + calls['c_ask']) / 2

calls['moneyness'] = calls['underlying_last'] / calls['strike']

print('First date:', calls['quote_date'].min().date())
print('Last date: ', calls['quote_date'].max().date())
print('Trading days:', calls['quote_date'].nunique())

calls[['mid', 'underlying_last', 'strike', 'moneyness', 'dte']].describe()

NameError: name 'calls' is not defined

In [11]:
problems = pd.Series({
    'missing bid or ask':     calls[['c_bid', 'c_ask']].isna().any(axis=1).sum(),
    'bid = 0':                (calls['c_bid'] == 0).sum(),
    'ask < bid (crossed)':    (calls['c_ask'] < calls['c_bid']).sum(),
    'expiring today (DTE<1)': (calls['dte'] < 1).sum(),
    'duplicate rows':         calls.duplicated(['quote_date', 'expire_date', 'strike']).sum(),
})
problems_table = pd.DataFrame({'rows': problems, '% of all': (100 * problems / len(calls)).round(2)})
problems_table.to_csv(PROCESSED_DIR / 'week1_problem_counts.csv')
problems_table

NameError: name 'calls' is not defined

In [12]:
coverage = calls.groupby('year').agg(
    rows=('mid', 'size'),
    trading_days=('quote_date', 'nunique'),
    first_date=('quote_date', 'min'),
    last_date=('quote_date', 'max'),
)
print(coverage)

calls['moneyness_bucket'] = pd.cut(calls['moneyness'], [0, 0.8, 0.9, 0.97, 1.03, 1.1, 1.2, np.inf],
                                   labels=['<0.80', '0.80-0.90', '0.90-0.97', '0.97-1.03 ATM',
                                           '1.03-1.10', '1.10-1.20', '>1.20'])
calls['dte_bucket'] = pd.cut(calls['dte'], [-1, 7, 30, 90, 180, 365, np.inf],
                             labels=['0-7d', '8-30d', '31-90d', '91-180d', '181-365d', '>1y'])

pd.crosstab(calls['moneyness_bucket'], calls['dte_bucket'], margins=True)

NameError: name 'calls' is not defined

In [13]:
years = sorted(calls['year'].unique())
TEST_YEAR  = years[-1]                                    # most recent year = final test
TEST_START = pd.Timestamp(f'{TEST_YEAR}-01-01')
VAL_START  = TEST_START - pd.DateOffset(months=3)         # last 3 months before the test year

if len(years) == 1:                                       # only one year: Jan-Aug / Sep / Oct-Dec
    VAL_START  = pd.Timestamp(f'{TEST_YEAR}-09-01')
    TEST_START = pd.Timestamp(f'{TEST_YEAR}-10-01')

calls['split'] = 'train'
calls.loc[calls['quote_date'] >= VAL_START,  'split'] = 'validation'
calls.loc[calls['quote_date'] >= TEST_START, 'split'] = 'test'

split_table = calls.groupby('split')['quote_date'].agg(first='min', last='max', days='nunique', rows='size')
split_table = split_table.reindex(['train', 'validation', 'test'])
split_table

NameError: name 'calls' is not defined